# Setup

In [88]:


import time
import chromadb

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_core.documents import Document

from langchain_chroma import Chroma

from datetime import datetime



In [89]:
from dotenv import load_dotenv
load_dotenv()
import os
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')

Below is the api key, endpoint, api version, deployment name for the chat model

In [90]:
from groq import Groq

client = Groq()

In [500]:
model_name = 'llama-3.1-8b-instant'

 Instantiating the embedding model

In [92]:
### Before running the command ensure the numpy version is 2.3.4. If not then upgrade numpy as per previous steps in notebook.
### Then do not restart the notebook. If you restart then you will loose all loaded variables. Just click Cancel instad of Restart

from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

# Download Raw Data


In [ ]:
import zipfile
 
zip_path = r"C:\Users\Aditya Sodani\Desktop\assess-2\AI_Training_Batch_May_2026\submissions\assignments\assignment-02\aditya-sodani\tesla-annual-reports.zip"
extract_to = r"C:\Users\Aditya Sodani\Desktop\assess-2\AI_Training_Batch_May_2026\submissions\assignments\assignment-02\aditya-sodani"
 
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)
 
print("Unzipped successfully!")

# Chunking

In [93]:
from pathlib import Path
pdf_folder_location = Path("tesla-annual-reports").resolve()
print("Loading PDFs from:", pdf_folder_location)

Loading PDFs from: C:\Users\Aditya Sodani\Desktop\assess-2\AI_Training_Batch_May_2026\submissions\assignments\assignment-02\aditya-sodani\tesla-annual-reports


In [94]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
pdf_loader = PyPDFDirectoryLoader(str(pdf_folder_location))

In [95]:
type(pdf_loader)

langchain_community.document_loaders.pdf.PyPDFDirectoryLoader

In [96]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=16
)

In [97]:
tesla_10k_chunks = pdf_loader.load_and_split(text_splitter)

In [98]:
len(tesla_10k_chunks)

3337

In [99]:
print(tesla_10k_chunks[0])

page_content='UNITED	STATES
SECURITIES	AND	EXCHANGE	COMMISSION
Washington,	D.C.	20549
	
FORM	
10-K/A
(Amendment	No.	1)
	
(Mark	One)
☒
ANNUAL	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	fiscal	year	ended	
December	31,	
2021
	
OR
☐
TRANSITION	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	transition	period	from	
																				
	to	
																					
Commission	File	Number:	
001-34756
	
Tesla,	Inc.
(Exact	name	of	registrant	as	specified	in	its	charter)
	
	
Delaware
	
91-2197729
(State	or	other	jurisdiction	of
incorporation	or	organization)
	
(I.R.S.	Employer
Identification	No.)
	
	
	
1	Tesla	Road
Austin
,	
Texas
	
78725
(Address	of	principal	executive	offices)
	
(Zip	Code)
(
512
)	
516-8177
(Registrant’s	telephone	number,	including	area	code)
Securities	registered	pursuant	to	Section	12(b)	of	the	Act:
	
Title	of	each	class
Trading	Symbol(s)
Name	of	each	exchange	on	which	registered
Common	stock
TSLA

In [100]:
print(tesla_10k_chunks[0].page_content)

UNITED	STATES
SECURITIES	AND	EXCHANGE	COMMISSION
Washington,	D.C.	20549
	
FORM	
10-K/A
(Amendment	No.	1)
	
(Mark	One)
☒
ANNUAL	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	fiscal	year	ended	
December	31,	
2021
	
OR
☐
TRANSITION	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	transition	period	from	
																				
	to	
																					
Commission	File	Number:	
001-34756
	
Tesla,	Inc.
(Exact	name	of	registrant	as	specified	in	its	charter)
	
	
Delaware
	
91-2197729
(State	or	other	jurisdiction	of
incorporation	or	organization)
	
(I.R.S.	Employer
Identification	No.)
	
	
	
1	Tesla	Road
Austin
,	
Texas
	
78725
(Address	of	principal	executive	offices)
	
(Zip	Code)
(
512
)	
516-8177
(Registrant’s	telephone	number,	including	area	code)
Securities	registered	pursuant	to	Section	12(b)	of	the	Act:
	
Title	of	each	class
Trading	Symbol(s)
Name	of	each	exchange	on	which	registered
Common	stock
TSLA
The	Nasdaq	Gl

# Database Creation

In [501]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [102]:
chromadb_client.heartbeat()

1780546532782389300

In [103]:
chromadb_client

In [104]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

In [105]:
vectorstore = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [106]:
chromadb_client.count_collections()

1

In [ ]:
i = 0 # Initialize the starting index for the chunks

while i < len(tesla_10k_chunks): # Iterate while the index is less than the total number of chunks
    vectorstore.add_documents( # Add documents to the vector store in batches of 500
        documents=tesla_10k_chunks[i:i+500], # Get the current batch of 500 chunks
        ids=["text_" + str(i) for i in range(i, i+500)] # Assign unique IDs to each chunk in the batch
    )

    i += 500 # Increment the index by 500 to move to the next batch
    time.sleep(1) # Pause for 30 seconds to avoid rate limiting issues with the vector store

In [502]:
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [503]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [504]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001D1A7DCC790>, search_kwargs={'k': 5})

## Basic RAG

In [505]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

In [506]:
qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

#Q1

In [507]:
user_query = "Does Tesla's growth story appear more constrained by external supply risk or internal execution and cost structure? Use evidence across Risk Factors and MD&A. "

#Q2

In [378]:
user_query="Explain how Tesla's AI and product roadmap is reflected in spending, operational priorities, and risk disclosures. Avoid generic AI claims." 

#Q3

In [440]:
user_query="A supplier asks whether Tesla is exposed to concentration risk across factories, suppliers, raw materials, or geographies. Prepare a concise risk note with citations. "

#Q4

In [471]:
user_query="Compare the strategic importance of automotive operations and energy generation/storage using evidence from the 10-K. What disclosures are needed to support a business recommendation? "

In [508]:

model_name = 'openai/gpt-oss-120b'

docs = retriever.invoke(user_query)
#docs = vectorstore_persisted.similarity_search_with_score(user_query, k=5)
context_list = [doc.page_content for doc in docs]
#context_list = [d.page_content for d in docs]
context_for_query = "\n---\n".join(context_list)

prompt = [
    {'role': 'system', 'content': qna_system_message},
    {'role': 'user', 'content': qna_user_message_template.format(
         context=context_for_query,
         question=user_query
        )
    }
]


try:
    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0.2
    )

    prediction = response.choices[0].message.content.strip()
except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)

The material that is provided highlights internal execution challenges—specifically, “any delays in scaling manufacturing, delivery and service operations to meet demand” and concerns about quarterly production and sales performance versus market expectations. While it also notes external factors such as competition and uncertainty about the future of electric vehicles, the emphasis is on Tesla’s ability to execute its manufacturing and operational plans rather than on external supply‑chain constraints or cost‑structure issues. Therefore, based on the Risk Factors and MD&A excerpts available, Tesla’s growth story appears more constrained by internal execution than by external supply risk.


In [ ]:
baseline_top_chunks = []

for idx, doc in enumerate(docs):

    baseline_top_chunks.append({
        "chunk_id": f"chunk_{idx+1}",
        "section": "Unknown",
        "year": 2023
    })

# RAG With Query Expansion Technique

In [510]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

In [511]:
query_expansion_system_message = """
You are an financial domain expert assisting in answering questions related to 10-k reports.
Perform query expansion on the question below. If there are multiple common ways of phrasing a user question \
or common synonyms for key words in the question, make sure to return multiple versions \
of the query with the different phrasings.

If there are acronyms or words you are not familiar with, do not try to rephrase them.

Return at least 3 versions of the question as a list.
Generate only a list of questions, each question in a new line.
Do not number the list of questions or use bullet points.
Do not mention anything before or after the list.

"""

user_message_template="""
<Question>
{question}
</Question>
"""

In [512]:
model_name='openai/gpt-oss-120b'

In [513]:
prompt=[
    {'role':'system', 'content':query_expansion_system_message},
    {'role':'user',   'content': user_message_template.format(
        question=user_query
    )}
]

In [514]:
prompt

[{'role': 'system',
  'content': '\nYou are an financial domain expert assisting in answering questions related to 10-k reports.\nPerform query expansion on the question below. If there are multiple common ways of phrasing a user question or common synonyms for key words in the question, make sure to return multiple versions of the query with the different phrasings.\n\nIf there are acronyms or words you are not familiar with, do not try to rephrase them.\n\nReturn at least 3 versions of the question as a list.\nGenerate only a list of questions, each question in a new line.\nDo not number the list of questions or use bullet points.\nDo not mention anything before or after the list.\n\n'},
 {'role': 'user',
  'content': "\n<Question>\nDoes Tesla's growth story appear more constrained by external supply risk or internal execution and cost structure? Use evidence across Risk Factors and MD&A. \n</Question>\n"}]

In [515]:
query_expansions = client.chat.completions.create(model=model_name,
                                                  messages=prompt,
                                                  temperature=0)

In [516]:
type(query_expansions.choices[0].message.content)

str

In [517]:
print(query_expansions.choices[0].message.content)

Is Tesla's growth story limited more by external supply risks or by internal execution and cost structure, based on evidence from the Risk Factors and MD&A sections?  
Does the evidence in Tesla’s Risk Factors and MD&A suggest that external supply risk or internal execution and cost structure is the bigger constraint on its growth narrative?  
Which factor—external supply risk or internal execution and cost structure—appears to more restrict Tesla’s growth, according to the Risk Factors and MD&A disclosures?


In [518]:
print(query_expansions)

ChatCompletion(id='chatcmpl-656f262c-de21-4727-90b1-7897b2a59775', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Is Tesla's growth story limited more by external supply risks or by internal execution and cost structure, based on evidence from the Risk Factors and MD&A sections?  \nDoes the evidence in Tesla’s Risk Factors and MD&A suggest that external supply risk or internal execution and cost structure is the bigger constraint on its growth narrative?  \nWhich factor—external supply risk or internal execution and cost structure—appears to more restrict Tesla’s growth, according to the Risk Factors and MD&A disclosures?", role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='The user wants query expansion: produce multiple versions of the question. Provide at least 3 versions, each on new line, no numbering or bullets. Should keep the core meaning. Provide synonyms: "constrained by external supply

In [519]:
print(query_expansions.choices[-1].message.reasoning)

The user wants query expansion: produce multiple versions of the question. Provide at least 3 versions, each on new line, no numbering or bullets. Should keep the core meaning. Provide synonyms: "constrained by external supply risk or internal execution and cost structure", "limited by external supply chain risk vs internal operational execution and cost structure", "more hindered by external supply risks or internal execution and cost factors". Also mention "Risk Factors and MD&A". Provide at least 3. Ensure each line is a question. No extra text.


In [520]:
query_expansions_list = query_expansions.choices[0].message.content.strip().split("\n")

In [521]:
len(query_expansions_list)

3

In [522]:
query_expansions_list

["Is Tesla's growth story limited more by external supply risks or by internal execution and cost structure, based on evidence from the Risk Factors and MD&A sections?  ",
 'Does the evidence in Tesla’s Risk Factors and MD&A suggest that external supply risk or internal execution and cost structure is the bigger constraint on its growth narrative?  ',
 'Which factor—external supply risk or internal execution and cost structure—appears to more restrict Tesla’s growth, according to the Risk Factors and MD&A disclosures?']

In [523]:
expanded_context_list = []

In [524]:
# for query in query_expansions_list:
#     expanded_context_list.extend([d.page_content for d in retriever.invoke(query)])

In [525]:
expanded_context_list = []
expanded_top_chunks = []

for query in query_expansions_list:

    # docs = vectorstore_persisted.similarity_search_with_score(
    #     query,
    #     k=3
    # )
      docs = retriever.invoke(user_query)
      for doc in docs:

        expanded_context_list.append(
            doc.page_content
        )

        expanded_top_chunks.append({
            "chunk_id": f"chunk_{doc.metadata['page']}",
            "section": "Unknown",
            "year": 2023,
            "retrieved_by": query
        })

In [526]:
print(expanded_context_list)

['developing\tbusiness\trelationships\twith\tus\tif\tthey\tare\tnot\tconvinced\tthat\tour\tbusiness\twill\tsucceed.\tAccordingly,\tin\torder\tto\tbuild\tand\tmaintain\tour\nbusiness,\twe\tmust\tmaintain\tconfidence\tamong\tcustomers,\tsuppliers,\tanalysts,\tratings\tagencies\tand\tother\tparties\tin\tour\tlong-term\tfinancial\tviability\tand\nbusiness\tprospects.\tMaintaining\tsuch\tconfidence\tmay\tbe\tparticularly\tcomplicated\tby\tcertain\tfactors\tincluding\tthose\tthat\tare\tlargely\toutside\tof\tour\ncontrol,\tsuch\tas\tour\tlimited\toperating\thistory,\tcustomer\tunfamiliarity\twith\tour\tproducts,\tany\tdelays\tin\tscaling\tmanufacturing,\tdelivery\tand\tservice\noperations\tto\tmeet\tdemand,\tcompetition\tand\tuncertainty\tregarding\tthe\tfuture\tof\telectric\tvehicles\tor\tour\tother\tproducts\tand\tservices,\tand\tour\tquarterly\nproduction\tand\tsales\tperformance\tcompared\twith\tmarket\texpectations.\nIn\tparticular,\tTesla’s\tproducts,\tbusiness,\tresults\tof\toperations

In [527]:
citations = []

seen = set()

for chunk in expanded_top_chunks:

    if chunk["chunk_id"] not in seen:

        seen.add(chunk["chunk_id"])

        citations.append({
            "chunk_id": chunk["chunk_id"],
            "source_doc": "tsla-20231231-gen.pdf",
            "section": chunk["section"],
            "year": chunk["year"]
        })

In [528]:
len(expanded_context_list)

9

In [529]:
expanded_context_list

['developing\tbusiness\trelationships\twith\tus\tif\tthey\tare\tnot\tconvinced\tthat\tour\tbusiness\twill\tsucceed.\tAccordingly,\tin\torder\tto\tbuild\tand\tmaintain\tour\nbusiness,\twe\tmust\tmaintain\tconfidence\tamong\tcustomers,\tsuppliers,\tanalysts,\tratings\tagencies\tand\tother\tparties\tin\tour\tlong-term\tfinancial\tviability\tand\nbusiness\tprospects.\tMaintaining\tsuch\tconfidence\tmay\tbe\tparticularly\tcomplicated\tby\tcertain\tfactors\tincluding\tthose\tthat\tare\tlargely\toutside\tof\tour\ncontrol,\tsuch\tas\tour\tlimited\toperating\thistory,\tcustomer\tunfamiliarity\twith\tour\tproducts,\tany\tdelays\tin\tscaling\tmanufacturing,\tdelivery\tand\tservice\noperations\tto\tmeet\tdemand,\tcompetition\tand\tuncertainty\tregarding\tthe\tfuture\tof\telectric\tvehicles\tor\tour\tother\tproducts\tand\tservices,\tand\tour\tquarterly\nproduction\tand\tsales\tperformance\tcompared\twith\tmarket\texpectations.\nIn\tparticular,\tTesla’s\tproducts,\tbusiness,\tresults\tof\toperations

In [530]:
final_context_documents = set(expanded_context_list)

In [531]:
len(final_context_documents)

3

In [532]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

In [533]:
qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [534]:
model_name = 'openai/gpt-oss-120b'

prompt = [
    {'role': 'system', 'content': qna_system_message},
    {'role': 'user', 'content': qna_user_message_template.format(
         context=final_context_documents,
         question=user_query
        )
    }
]

try:
    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()
except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)


I don't know.


In [499]:
import json
import os
 
json_file = "query_expansion_results.json"
 
if os.path.exists(json_file):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)
 
    if not isinstance(data, list):
        data = [data]
else:
    data = []
 
question_id = f"Q{len(data) + 1}"
 
output_schema = {
    "question_id": question_id,
    "original_query": user_query,
    "expanded_queries": query_expansions_list,
    "baseline_top_chunks": baseline_top_chunks,
    "expanded_top_chunks": expanded_top_chunks,
    "final_answer": prediction,
    "citations": citations,
    "retrieval_improvement_analysis":
        "Query expansion improved retrieval coverage compared to baseline retrieval."
}
 
data.append(output_schema)
 
with open(json_file, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4, ensure_ascii=False)
 